In [ ]:
## import all packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq
import os
import random
import warnings
import importlib
warnings.filterwarnings('ignore')

print(f"NumPy   version : {np.__version__}")
print(f"Pandas  version : {pd.__version__}")
print(f"Scanpy  version : {sc.__version__}")
print(f"Squidpy version : {sq.__version__}")


In [ ]:
## creating necessary dirs
os.makedirs('./data/raw', exist_ok=True)
os.makedirs('./results/figures', exist_ok=True)
os.makedirs('./results/tables',  exist_ok=True)
os.makedirs('./data/temp',  exist_ok=True)

In [ ]:
## settings for scanpy
sc.settings.verbosity = 3          
sc.settings.figdir   = '/results/figures'
sc.settings.set_figure_params(
    dpi=120,           
    facecolor='white', 
    fontsize=12
)

SEED = 2 ## setting a random seed for future reproducbiltu across scochrastic processes
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
## add the wget commands to download the data here

In [ ]:
adata = sc.read_visium(
    path='./data/raw/',
    count_file='Visium_FFPE_Human_Breast_Cancer_filtered_feature_bc_matrix.h5'
)

adata.var_names_make_unique()

In [ ]:
adata

In [ ]:
## store raw counts for downstream use such as DEG
adata.layers['counts'] = adata.X.copy()

adata

In [ ]:
print(adata.obsm['spatial'].shape)

In [ ]:
print(adata.uns['spatial'].keys()) ## images available

In [ ]:
## remoce mito genes, use mt- if you are using mouse data
adata.var['mt'] = adata.var_names.str.startswith('MT-')
print(f"mito genes: {adata.var['mt'].sum()}")

In [ ]:
## per spot quality metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=['mt'],   
    inplace=True,     
    log1p=True      
)

In [ ]:

## plots for qc metrics
sc.pl.violin(
    adata,
    keys=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True,
    save='_01_QC_violin.png'
)

sc.pl.scatter(
    adata,
    x='total_counts', y='n_genes_by_counts',
    color='pct_counts_mt',
    save='_01_QC_scatter.png'
)

In [ ]:
## filtering the data
print(f"spots before filtering: {adata.n_obs:,}")
print(f"genes before filtering: {adata.n_vars:,}")

sc.pp.filter_cells(adata, min_counts=500)   
sc.pp.filter_cells(adata, min_genes=200)    
sc.pp.filter_genes(adata, min_cells=10)     
adata = adata[adata.obs.pct_counts_mt < 15] 

print(f"spots after filtering: {adata.n_obs:,}")
print(f"genes after filtering: {adata.n_vars:,}")

In [ ]:
## plots for qc metrics after filtering
sc.pl.violin(
    adata,
    keys=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True,
    save='_02_post_filtering_QC_violin.png'
)

sc.pl.scatter(
    adata,
    x='total_counts', y='n_genes_by_counts',
    color='pct_counts_mt',
    save='_02_post_filtering_QC_scatter.png'
)

In [ ]:
## normalization to 10,000 counts per spot to correct for diff in sequencing depth across spots
sc.pp.normalize_total(
    adata,
    inplace=True,
    target_sum=1e4  
)

## log transform
sc.pp.log1p(adata)

## save them in a separate layer 
adata.layers['log1p_norm'] = adata.X.copy()

In [ ]:
## hvg selection - basically identify those genes with high variability across spots
sc.pp.highly_variable_genes(
    adata,
    flavor='seurat',
    n_top_genes=3000,    
    inplace=True         ## add a boolean column to adata.var
)

## plot to see the mean-variance relationship
sc.pl.highly_variable_genes(
    adata,
    save='_03_HVGs.png'
)


In [ ]:
## scale the data
sc.pp.scale(adata, max_value=10)

In [ ]:
## pca
sc.tl.pca(
    adata,
    n_comps=50,
    svd_solver='arpack',
    use_highly_variable=True  ## only use HVGs not all the genes for PCA
)
## elbow plot - used to select the ideal number of PC's for clustering
sc.pl.pca_variance_ratio(
    adata,
    n_pcs=50,
    log=True,               # log scale for better visualization
    save='_03_PCA_elbow.png'
)

In [ ]:
## build the neighborhood graph in PCA space - using 30 PC's - whatever the eblow point is 
sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=40,
    random_state=2
)

In [ ]:
## umap projection to see in 2D - a non-lineaar dimensionality reduction method
sc.tl.umap(adata, random_state=2)
print(f"pca shape: {adata.obsm['X_pca'].shape}")
print(f"umap shape: {adata.obsm['X_umap'].shape}")

In [ ]:
## leiden clustering 
sc.tl.leiden(
    adata,
    resolution=1.0, ## adjust as needed for granularity or coarseness
    random_state=2,
    key_added='leiden'   
)

In [ ]:
n_clusters = adata.obs['leiden'].nunique()
print(f"Clusters found at resolution=0.5 : {n_clusters}")
print("\nSpots per cluster:")
print(adata.obs['leiden'].value_counts().sort_index())

In [ ]:
## umap visualization of the clusters
sc.pl.umap(
    adata,
    color=['leiden', 'total_counts', 'n_genes_by_counts'],
    ncols=3,
    legend_loc='on data',  # show cluster labels on plot
    title=['Leiden Clusters', 'Total UMI Counts', 'Genes Detected'],
    save='_05_umap_clusters.png'
)

## visualize on the tissue sample prsnt 
sc.pl.spatial(
    adata,
    img_key='hires',
    color=['leiden'],
    size=1.5,              
    legend_loc='right margin',
    title='Spatial Clusters - Human Breast Cancer',
    save='_06_clusters_on_tissue.png'
)

## visualize the expression of the marker genes on the tissue sample 
sc.pl.spatial(
    adata,
    img_key='hires',
    color=['EPCAM', 'MKI67', 'COL1A1', 'CD68', 'CD3D', 'PECAM1'],
    size=1.5,
    ncols=3,
    cmap='Blues',           
    title=[
        'EPCAM (pan-epithelial / tumor marker)',
        'MKI67 (proliferating cells (S/G2/M phase))',
        'COL1A1 (collagen I, cancer-associated fibroblasts)',
        'CD68 (pan-macrophage markere)',
        'CD3D (pan-T cell marker (TCR complex))',
        'PECAM1 (endothelial cells (CD31))'
    ],
    save='_07_marker_genes_on_tissue.png'
)

In [ ]:
## wilcoxon rank-sum test: each cluster vs all others
## uses the log-normalized layer stored in adata.X
sc.tl.rank_genes_groups(
    adata,
    groupby  = 'leiden',
    method   = 'wilcoxon',
    pts      = True,       ## include fraction of spots expressing each gene
    key_added = 'de_wilcoxon',
    use_raw  = False
)

## extract into a tidy dataframe for inspection
de_df = sc.get.rank_genes_groups_df(adata, group=None, key='de_wilcoxon')
print(de_df.shape)
print(de_df.head(10).to_string(index=False))

In [ ]:
## dot plot: top 4 markers per cluster
## dot size = pct spots expressing gene, color = mean expression
sc.pl.rank_genes_groups_dotplot(
    adata,
    n_genes        = 4,
    key            = 'de_wilcoxon',
    groupby        = 'leiden',
    standard_scale = 'var',
    save           = '_08_dotplot_markers.png'
)

In [ ]:
## save full DE table for reference
de_df.to_csv('./results/tables/de_markers_per_cluster.csv', index=False)

## print top 3 DE genes per cluster - helpful for verifying cell type assignments below
print("Top 3 DE markers per cluster:")
for cl in sorted(de_df['group'].unique(), key=lambda x: int(x)):
    top3 = de_df[de_df['group'] == cl].head(3)['names'].tolist()
    print(f"  Cluster {cl:>2} : {', '.join(top3)}")

In [ ]:
## breast cancer TME cell type signature gene sets
## each list contains canonical markers from published atlases

breast_cancer_signatures = {
    'Tumor Epithelium': [
        'EPCAM', 'CDH1', 'KRT8', 'KRT18', 'KRT19', 'KRT7', 'CLDN4', 'MUC1'
    ],
    'Proliferating Tumor': [
        'MKI67', 'TOP2A', 'PCNA', 'STMN1', 'CDK1', 'CCNB1', 'UBE2C'
    ],
    'ER-positive Luminal': [
        'ESR1', 'PGR', 'FOXA1', 'GATA3', 'TFF1', 'CCND1', 'XBP1'
    ],
    'Cancer-Assoc. Fibroblasts': [
        'COL1A1', 'COL1A2', 'COL3A1', 'FAP', 'POSTN', 'ACTA2', 'VIM', 'S100A4'
    ],
    'Tumor-Assoc. Macrophages': [
        'CD68', 'CD163', 'MRC1', 'CSF1R', 'C1QA', 'APOE', 'LGMN'
    ],
    'CD8+ T Cells': [
        'CD8A', 'CD8B', 'GZMB', 'GZMK', 'PRF1', 'IFNG', 'NKG7'
    ],
    'CD4+ T Cells / Tregs': [
        'CD4', 'FOXP3', 'IL2RA', 'CTLA4', 'CD3D', 'CD3E', 'TIGIT'
    ],
    'B Cells': [
        'CD79A', 'MS4A1', 'CD19', 'IGHM', 'IGKC', 'BANK1'
    ],
    'Endothelial Cells': [
        'PECAM1', 'VWF', 'CDH5', 'CLDN5', 'ESAM', 'FLT1'
    ],
}

## check how many markers from each set are present in the dataset
print(f"{'Cell type':<35} Markers in dataset")
print("-" * 55)
for ct, genes in breast_cancer_signatures.items():
    present = [g for g in genes if g in adata.var_names]
    print(f"  {ct:<33} {len(present)}/{len(genes)}: {', '.join(present[:4])}...")

In [ ]:
import seaborn as sns

## score each spot against every cell type signature
## sc.tl.score_genes computes enrichment vs a random background set of equal size
score_keys = []
for cell_type, gene_list in breast_cancer_signatures.items():
    available = [g for g in gene_list if g in adata.var_names]
    if len(available) < 3:
        print(f"  [skip] {cell_type} — only {len(available)} genes found")
        continue
    key = 'score_' + cell_type.replace(' ', '_').replace('.', '').replace('+', 'pos').replace('/', '_')
    sc.tl.score_genes(adata, gene_list=available, score_name=key, random_state=2)
    score_keys.append((cell_type, key))
    print(f"  [ok] {cell_type}")

print(f"\nScored {len(score_keys)} cell types")

In [ ]:
## convert raw scores to pseudo-proportions using softmax normalization
## this ensures all proportions per spot sum to 1, making them interpretable
import numpy as np

score_matrix = adata.obs[[sk for _, sk in score_keys]].values.copy()

## shift scores to be positive before softmax (scores can be negative)
score_matrix = score_matrix - score_matrix.min(axis=1, keepdims=True)

## softmax: exp(x_i) / sum(exp(x_j))
exp_scores    = np.exp(score_matrix)
proportions   = exp_scores / exp_scores.sum(axis=1, keepdims=True)

## store in adata.obs with clean column names
prop_keys = []
for i, (ct, _) in enumerate(score_keys):
    prop_key = 'prop_' + ct.replace(' ', '_').replace('.', '').replace('+', 'pos').replace('/', '_')
    adata.obs[prop_key] = proportions[:, i]
    prop_keys.append((ct, prop_key))

print("Proportions stored in adata.obs")
print(f"Sum check (first 5 spots): {adata.obs[[pk for _, pk in prop_keys]].values[:5].sum(axis=1).round(4)}")

In [ ]:
## heatmap: mean deconvolved proportion per Leiden cluster
## shows which cell type dominates each cluster
prop_cluster_means = pd.DataFrame(index=sorted(adata.obs['leiden'].unique(), key=int))
for ct, prop_key in prop_keys:
    prop_cluster_means[ct] = adata.obs.groupby('leiden')[prop_key].mean()

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(
    prop_cluster_means.T,
    annot     = prop_cluster_means.T.round(2),
    fmt       = '.2f',
    cmap      = 'YlOrRd',
    linewidths = 0.4,
    linecolor  = 'white',
    ax        = ax,
    cbar_kws  = {'label': 'Mean Proportion (0–1)', 'shrink': 0.7},
    annot_kws = {'size': 7}
)
ax.set_xlabel('Leiden Cluster', fontsize=12)
ax.set_ylabel('Cell Type', fontsize=12)
ax.set_title('Cell Type Deconvolution — Mean Proportion per Cluster\nHuman Breast Cancer Visium (FFPE)', fontsize=13)
plt.tight_layout()
plt.savefig('./results/figures/_09_deconvolution_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## stacked bar chart: TME composition per cluster
fig, ax = plt.subplots(figsize=(14, 6))

colors = plt.cm.Set3(np.linspace(0, 1, len(prop_keys)))
bottom = np.zeros(len(prop_cluster_means))

for i, (ct, _) in enumerate(prop_keys):
    ax.bar(
        prop_cluster_means.index,
        prop_cluster_means[ct],
        bottom = bottom,
        label  = ct,
        color  = colors[i],
        edgecolor = 'white',
        linewidth = 0.5
    )
    bottom += prop_cluster_means[ct].values

ax.set_xlabel('Leiden Cluster', fontsize=12)
ax.set_ylabel('Estimated Cell Type Proportion', fontsize=12)
ax.set_title('Tumor Microenvironment Composition per Cluster', fontsize=13)
ax.legend(loc='upper right', bbox_to_anchor=(1.22, 1), fontsize=9)
plt.tight_layout()
plt.savefig('./results/figures/_10_tme_stacked_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## spatial maps: top 4 most abundant cell type proportions on tissue
top_types = prop_cluster_means.mean().nlargest(4).index.tolist()
top_prop_keys = [pk for ct, pk in prop_keys if ct in top_types]
top_labels    = [ct for ct, pk in prop_keys if ct in top_types]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, prop_key, label in zip(axes, top_prop_keys, top_labels):
    sc.pl.spatial(
        adata,
        img_key   = 'hires',
        color     = prop_key,
        color_map = 'RdYlBu_r',
        size      = 1.6,
        vmin      = 0,
        vmax      = 0.6,
        title     = label,
        ax        = ax,
        show      = False
    )

plt.suptitle('Deconvolved Cell Type Proportions on Tissue (Top 4)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('./results/figures/_11_deconvolution_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## PAGA: cluster connectivity graph
## shows which Leiden clusters are transcriptionally adjacent
sc.tl.paga(adata, groups='leiden')

fig, ax = plt.subplots(figsize=(8, 7))
sc.pl.paga(
    adata,
    threshold   = 0.05,   ## only draw edges with connectivity > 0.05
    node_size_scale = 1.5,
    edge_width_scale = 1.0,
    title       = 'PAGA — Cluster Connectivity Graph\nHuman Breast Cancer Visium',
    frameon     = False,
    ax          = ax,
    show        = False
)
plt.tight_layout()
plt.savefig('./results/figures/_12_paga_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print("PAGA connectivity matrix (threshold=0.05):")
print(adata.uns['paga']['connectivities'].toarray().round(2))

In [ ]:
## UMAP initialized with PAGA layout
## PAGA-initialized UMAP better preserves global cluster relationships
sc.tl.umap(adata, init_pos='paga', random_state=2)

sc.pl.umap(
    adata,
    color       = 'leiden',
    legend_loc  = 'on data',
    title       = 'UMAP (PAGA-initialized) — Leiden Clusters',
    save        = '_13_umap_paga_init.png'
)

In [ ]:

luminal_col = "prop_ER-positive_Luminal"

sc.pl.spatial(
    adata,
    color=[luminal_col],
    color_map="viridis",
    spot_size=1.0,
    save="_spatial_luminal_score.png",
)

In [ ]:
luminal_col = "prop_ER-positive_Luminal"

sc.pl.umap(
    adata,
    color=["leiden"],
    wspace=0.4,
    save="_umap_leiden.png",
)

sc.pl.umap(
    adata,
    color=[luminal_col],
    color_map="viridis",
    wspace=0.4,
    save="_umap_luminal_score.png",
)

In [ ]:
luminal_markers = ["EPCAM", "KRT8", "KRT18", "ESR1"]
basal_markers   = ["KRT5", "KRT14", "MKI67", "TOP2A"]
immune_markers  = ["CD3D", "CD3E", "CD8A", "FOXP3", "PTPRC"]
stroma_markers  = ["COL1A1", "COL1A2", "PDGFRB", "ACTA2"]

marker_genes = [g for g in (luminal_markers + basal_markers 
                            + immune_markers + stroma_markers)
                if g in adata.var_names]

sc.pl.dotplot(
    adata,
    var_names=marker_genes,
    groupby="leiden",
    dendrogram=True,
    standard_scale="var",
    swap_axes=True,
    save="_marker_dotplot_leiden.png",
)

In [ ]:
## set the root for pseudotime
## INSTRUCTIONS: change ROOT_CLUSTER to whichever cluster has the highest
## luminal/normal epithelial score from the deconvolution above.
## Look at prop_cluster_means['ER-positive Luminal'] and pick the top cluster.

luminal_scores = prop_cluster_means.get('ER-positive Luminal', prop_cluster_means.iloc[:, 0])
auto_root      = str(luminal_scores.idxmax())
print(f"Auto-selected root cluster: {auto_root}  (highest ER+ luminal proportion)")

# Manual override based on markers + spatial context:
# we choose cluster 7 as a luminal/normal-like root.
ROOT_CLUSTER = "7"
print(f"Using ROOT_CLUSTER = {ROOT_CLUSTER}  (manual override)")

## pick the medoid spot of the root cluster as the iroot
## (the spot closest to the cluster centroid in PCA space)
root_mask  = adata.obs['leiden'] == ROOT_CLUSTER
root_pca   = adata.obsm['X_pca'][root_mask]
centroid   = root_pca.mean(axis=0)
dists      = np.linalg.norm(root_pca - centroid, axis=1)
local_idx  = np.argmin(dists)
global_idx = np.where(root_mask)[0][local_idx]

adata.uns['iroot'] = int(global_idx)
print(f"Root spot index: {global_idx}  (cluster {ROOT_CLUSTER} medoid)")

In [ ]:
## compute diffusion map (required for DPT)
sc.tl.diffmap(adata, n_comps=15)

## diffusion pseudotime from the root spot
sc.tl.dpt(adata, n_dcs=10)
print("Pseudotime range:")
print(f"  min : {adata.obs['dpt_pseudotime'].min():.4f}")
print(f"  max : {adata.obs['dpt_pseudotime'].max():.4f}")
print(f"  mean: {adata.obs['dpt_pseudotime'].mean():.4f}")

In [ ]:
## UMAP colored by pseudotime
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(
    adata,
    color     = 'dpt_pseudotime',
    color_map = 'viridis',
    title     = 'UMAP — Diffusion Pseudotime',
    ax        = axes[0],
    show      = False
)

## mark the root spot on UMAP
root_umap = adata.obsm['X_umap'][adata.uns['iroot']]
axes[0].scatter(*root_umap, s=120, c='red', zorder=5, label='Root spot')
axes[0].legend(fontsize=9)

sc.pl.umap(
    adata,
    color     = 'leiden',
    legend_loc = 'on data',
    title     = 'UMAP — Leiden Clusters (reference)',
    ax        = axes[1],
    show      = False
)

plt.suptitle('Diffusion Pseudotime — Breast Cancer Visium', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('./results/figures/_14_pseudotime_umap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## spatial map: pseudotime painted on the H&E tissue
## reveals where early vs late transcriptional states physically reside
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

sc.pl.spatial(
    adata,
    img_key   = 'hires',
    color     = 'dpt_pseudotime',
    color_map = 'viridis',
    size      = 1.6,
    title     = 'Pseudotime on Tissue\n(dark=early, yellow=late)',
    ax        = axes[0],
    show      = False
)

sc.pl.spatial(
    adata,
    img_key   = 'hires',
    color     = 'leiden',
    size      = 1.5,
    legend_loc = 'right margin',
    title     = 'Leiden Clusters on Tissue',
    ax        = axes[1],
    show      = False
)

plt.suptitle('Spatial Pseudotime — Human Breast Cancer Visium (FFPE)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('./results/figures/_15_pseudotime_spatial.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
## gene expression along pseudotime trajectory
## bin spots into 20 pseudotime windows and plot mean expression
## of key genes expected to change during tumor progression

trajectory_genes = ['EPCAM', 'MKI67', 'COL1A1', 'CD68', 'FOXP3', 'ESR1', 'ERBB2', 'CDH1']
trajectory_genes = [g for g in trajectory_genes if g in adata.var_names]

## restore lognorm expression for plotting (adata.X is scaled at this point)
expr_matrix = pd.DataFrame(
    adata.layers['log1p_norm'][:, [adata.var_names.get_loc(g) for g in trajectory_genes]].toarray()
    if hasattr(adata.layers['log1p_norm'], 'toarray')
    else adata.layers['log1p_norm'][:, [adata.var_names.get_loc(g) for g in trajectory_genes]],
    index   = adata.obs_names,
    columns = trajectory_genes
)
expr_matrix['pseudotime'] = adata.obs['dpt_pseudotime'].values

## bin into 20 equal pseudotime windows
expr_matrix['pt_bin'] = pd.cut(expr_matrix['pseudotime'], bins=20, labels=False)
binned = expr_matrix.groupby('pt_bin')[trajectory_genes].mean()

## smooth with a rolling window to reduce noise
smoothed = binned.rolling(window=3, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(trajectory_genes)))
for gene, col in zip(trajectory_genes, colors):
    ax.plot(smoothed.index, smoothed[gene], label=gene, color=col, lw=2.0)

ax.set_xlabel('Pseudotime Bin (early → late)', fontsize=12)
ax.set_ylabel('Mean Log-normalized Expression', fontsize=12)
ax.set_title('Gene Expression Along Pseudotime Trajectory\nHuman Breast Cancer', fontsize=13)
ax.legend(loc='upper right', fontsize=9)
ax.axvline(0, color='red', lw=1.5, linestyle='--', alpha=0.5, label='Root')
plt.tight_layout()
plt.savefig('./results/figures/_16_pseudotime_gene_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
adata.obs['dpt_pseudotime'] = adata.obs['dpt_pseudotime'].astype(float)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 4))
sns.scatterplot(
    x=adata.obs['dpt_pseudotime'],
    y=adata.obs['prop_ER-positive_Luminal'],
    s=10,
    alpha=0.5,
    ax=ax,
)
ax.set_xlabel("Diffusion pseudotime")
ax.set_ylabel("prop_ER-positive_Luminal")
ax.set_title("Luminal score vs pseudotime")
plt.tight_layout()
plt.savefig("figures/pseudotime_vs_luminal.png", dpi=200)

In [ ]:
## save the fully annotated AnnData
adata.write('./results/breast_cancer_spatial_analyzed.h5ad')


print("  Analysis complete!")

print(f"  Spots  : {adata.n_obs}")
print(f"  Genes  : {adata.n_vars}")
print(f"  Clusters : {adata.obs['leiden'].nunique()}")
print(f"  Outputs  : ./results/figures/  and  ./results/tables/")

### Diffusion pseudotime summary

- Root: Leiden cluster 7 (luminal/normal-like epithelium), chosen based on
  deconvolution (`prop_ER-positive_Luminal`), marker expression, and PAGA connectivity.
- Direction: early → late roughly follows 7 → 6/3 → 5/8/10, corresponding to
  a transition from luminal/normal-like regions toward more invasive,
  ERBB2-high tumor regions.
- Along pseudotime, stromal marker COL1A1 decreases, while epithelial/luminal
  markers (EPCAM, CDH1, ERBB2) increase, consistent with tumor progression.